## **MRMS RadarOnly QPE**

### **Imports**

In [14]:

from pathlib import Path
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import random
from datetime import datetime, timedelta
import gzip
import shutil


In [21]:
random.seed(14912)

### **Paths**

In [8]:

root_path = Path.cwd()
outdir_path = root_path / 'data/samples'
outdir_path.mkdir(parents=True, exist_ok=True)

print(outdir_path.exists())


True



### **MRMS**

The **Multi-Radar/Multi-Sensor (MRMS)** system is an operational NOAA/NWS framework that combines weather radar, environmental information, and other observations to generate high-resolution precipitation and severe-weather products.


### **RadarOnly QPE**

This is the primary product used in this project and it represents **radar-derived precipitation accumulated over the previous one-hour period**, without the gauge-bias correction applied to other MRMS gauge-adjusted QPE products.

| Property              | `RadarOnly_QPE_01H`                      |
| --------------------- | ---------------------------------------- |
| Product name          | `RadarOnly_QPE_01H`                      |
| Physical quantity     | Radar-derived precipitation accumulation |
| Accumulation period   | 1 hour                                   |
| Update interval       | 2 minutes                                |
| Units                 | mm                                       |
| Spatial resolution    | ~1 km × 1 km                             |
| Main upstream product | Surface Precipitation Rate (SPR)         |
| Gauge bias correction | No                                       |
| Format                | GRIB2                                    |
| Primary project role  | Base radar precipitation observation     |


The aim of this project is to forecast/predict the hourly corrected precipitation values and therefore we will be using `RadarOnly_QPE_01H` as one of our primary inputs. It represents a **1-hour precipitation accumulation**, but a new product is generated approximately every **2 minutes**. Therefore, consecutive files largely represent **overlapping rolling one-hour accumulation windows**.

For example:

```text id="j3gvfe"
12:00 file → precipitation accumulated approximately 11:00–12:00
12:02 file → precipitation accumulated approximately 11:02–12:02
12:04 file → precipitation accumulated approximately 11:04–12:04
```

Thus, the files should **not simply be summed** to calculate longer-duration precipitation because most of their accumulation periods overlap.




### **RadarOnly QPE Can Be Unreliable**

Radar-derived precipitation estimates can contain substantial errors even when a valid numerical precipitation value is present. Important sources of uncertainty include:

* **Beam blockage:** Terrain can partially or completely block the radar beam.
* **Radar range:** At greater distances, the radar beam samples increasingly higher elevations above the surface.
* **Precipitation microphysics:** Rain, snow, hail, and mixed precipitation produce different radar responses.
* **Ground clutter:** Terrain, buildings, and other non-meteorological objects can contaminate radar observations.
* **Anomalous propagation:** Atmospheric conditions can bend the radar beam abnormally and produce non-meteorological echoes.
* **Hail:** Strong radar reflectivity from hail can lead to substantial precipitation overestimation.
* **Attenuation:** Heavy precipitation can weaken the radar signal, potentially causing underestimation farther along the radar beam.

These limitations are particularly important because a **numerically valid RadarOnly QPE value does not necessarily imply that the estimate is equally reliable at every location or time**.


Now lets work on downloading a few files and we will inspect some of the fields such as metadata, units and value ranges etc. 

---


### **Data Download**

The 1H data are publicly available through the [**NOAA MRMS Amazon S3 archive**](https://noaa-mrms-pds.s3.amazonaws.com/index.html). It has the structure:

```text id="2x2pbb"
CONUS/
└── RadarOnly_QPE_01H_00.00/
    ├── YYYYMMDD/
    │   ├── MRMS_RadarOnly_QPE_01H_00.00_YYYYMMDD-HHMMSS.grib2.gz
    │   └── ...
```

For example:

```text id="r8bhjm"
MRMS_RadarOnly_QPE_01H_00.00_20201014-000000.grib2.gz
MRMS_RadarOnly_QPE_01H_00.00_20201014-000200.grib2.gz
MRMS_RadarOnly_QPE_01H_00.00_20201014-000400.grib2.gz
```

The timestamps increase by approximately **2 minutes**, consistent with the expected update frequency of the product. Therefore a complete would theoretically contain:

$$
\frac{24 \times 60}{2} = 720\ \text{files/day}
$$

However, not every day will have 720 files, some days will have missing files. We will verify the daily files count later once we download the complete dataset. 

At this stage, we wont be downloading complete data. We will download a small number of representative `.grib2.gz` files and inspect the data values, distribution and visual a few days data. Lets get started. 


In [24]:

# NOAA MRMS public S3 bucket
bucket = "noaa-mrms-pds"
product = "CONUS/RadarOnly_QPE_01H_00.00" # see the above structure

# we can download a random date data between October 16 to onward - 

start_date = datetime(2020, 10, 16)
end_date = datetime.today() - timedelta(days=1)

# get a random date
random_days = random.randint(0, (end_date - start_date).days)
random_date = (start_date + timedelta(days=random_days)).strftime("%Y%m%d")
random_date


'20220820'

In [25]:

# we will use boto3, no need for any credentials, anonymous access
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

prefix = f"{product}/{random_date}/"

# lets see available files for the selected date
response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)

files = [obj["Key"] for obj in response.get("Contents", []) if obj["Key"].endswith(".grib2.gz")]

print(f"Number of files found: {len(files)}")

# print a few files paths
for key in files[:10]:
    print(key)
    

Number of files found: 720
CONUS/RadarOnly_QPE_01H_00.00/20220820/MRMS_RadarOnly_QPE_01H_00.00_20220820-000000.grib2.gz
CONUS/RadarOnly_QPE_01H_00.00/20220820/MRMS_RadarOnly_QPE_01H_00.00_20220820-000200.grib2.gz
CONUS/RadarOnly_QPE_01H_00.00/20220820/MRMS_RadarOnly_QPE_01H_00.00_20220820-000400.grib2.gz
CONUS/RadarOnly_QPE_01H_00.00/20220820/MRMS_RadarOnly_QPE_01H_00.00_20220820-000600.grib2.gz
CONUS/RadarOnly_QPE_01H_00.00/20220820/MRMS_RadarOnly_QPE_01H_00.00_20220820-000800.grib2.gz
CONUS/RadarOnly_QPE_01H_00.00/20220820/MRMS_RadarOnly_QPE_01H_00.00_20220820-001000.grib2.gz
CONUS/RadarOnly_QPE_01H_00.00/20220820/MRMS_RadarOnly_QPE_01H_00.00_20220820-001200.grib2.gz
CONUS/RadarOnly_QPE_01H_00.00/20220820/MRMS_RadarOnly_QPE_01H_00.00_20220820-001400.grib2.gz
CONUS/RadarOnly_QPE_01H_00.00/20220820/MRMS_RadarOnly_QPE_01H_00.00_20220820-001600.grib2.gz
CONUS/RadarOnly_QPE_01H_00.00/20220820/MRMS_RadarOnly_QPE_01H_00.00_20220820-001800.grib2.gz


In [26]:
response.keys()

dict_keys(['ResponseMetadata', 'IsTruncated', 'Contents', 'Name', 'Prefix', 'MaxKeys', 'EncodingType', 'KeyCount'])



# References

**[1] NOAA — MRMS Dataset, Registry of Open Data on AWS**
Provides the official MRMS overview, operational history, two-minute update cycle, v11→v12 transition, and public AWS access information.
https://registry.opendata.aws/noaa-mrms-pds/

**[2] NOAA Warning Decision Training Division — Surface Precipitation Rate (SPR)**
Describes SPR, its approximately 1-km spatial resolution, two-minute temporal resolution, and its use in constructing one-hour RadarOnly QPE accumulations.
https://vlab.noaa.gov/web/wdtd/-/surface-precipitation-rate-spr-1

**[3] NOAA Warning Decision Training Division — Seamless Hybrid Scan Reflectivity (SHSR)**
Describes the quality-controlled low-level reflectivity field used as an input to Surface Precipitation Rate and its relationship with radar blockage information.
https://vlab.noaa.gov/web/wdtd/-/seamless-hybrid-scan-reflectivity-shsr-

**[4] NOAA Warning Decision Training Division — Surface Precipitation Type (SPT)**
Describes precipitation-type classification and its role in selecting radar precipitation relationships for Surface Precipitation Rate.
https://vlab.noaa.gov/web/wdtd/-/surface-precipitation-type-spt-

**[5] NOAA/NSSL — Operational MRMS GRIB2 Product Tables**
Official product table listing `RadarOnly_QPE_01H`, its one-hour accumulation period, two-minute update interval, units (`mm`), and MRMS GRIB2 metadata/special codes.
https://www.nssl.noaa.gov/projects/mrms/operational/tables.php

**[6] NOAA Warning Decision Training Division — QPE: Radar Only**
Official training/documentation page describing the MRMS RadarOnly precipitation accumulation product and its characteristics.
https://vlab.noaa.gov/web/wdtd/-/qpe-radar-only

**[7] NOAA/NSSL — Timeline of MRMS Builds**
Documents the October 2020 MRMS v12 transition and major QPE-related changes, including updates to RadarOnly QPE and introduction of RAQI.
https://inside.nssl.noaa.gov/mrms/build_timeline/

**[8] NOAA/NSSL — Multi-Radar/Multi-Sensor System**
General official MRMS system overview and documentation entry point.
https://www.nssl.noaa.gov/projects/mrms/

**[9] NOAA MRMS Public AWS Archive**
Primary source from which the raw MRMS files used in this notebook will be downloaded.
https://noaa-mrms-pds.s3.amazonaws.com/index.html
